# Diffusion: Clear vs. Disruption Visualization

Generate and visualize samples from the AdaLN-conditioned diffusion model:
- **Condition 1**: Clear (class_id=0) vs. Disruption (class_id=1)
- **Condition 2**: For disruption, `t_disrupt` = normalised (t_disruption - 300ms)

Contrast generated **clear** vs **disruption** samples as 2D images (channels × time).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt
import numpy as np

from diffusion.models import UNet2DAdaLN, DDPMScheduler
from diffusion.data.dataset import DecimatedEceiMmapDataset

## Load model and scheduler

In [ ]:
CHECKPOINT = Path("../checkpoints/ckpt_epoch_100.pt")  # or latest
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
config = ckpt.get("config", {})
base_channels = config.get("base_channels", 64)
num_timesteps = config.get("num_timesteps", 1000)

model = UNet2DAdaLN(
    in_channels=1, out_channels=1, base_channels=base_channels,
    channel_mults=(1, 2, 4, 8), num_classes=2, time_embed_dim=128, cond_embed_dim=128,
).to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
scheduler = DDPMScheduler(num_timesteps=num_timesteps).to(DEVICE)
print("Model loaded.")

## Generate Clear vs Disruption samples

In [ ]:
n_samples = 4
shape = (n_samples, 1, 160, 7512)

samples_clear = None
samples_disrupt = None

with torch.no_grad():
    # Clear (class_id=0, t_disrupt=0 unused)
    cond_clear = {"class_id": torch.zeros(n_samples, device=DEVICE, dtype=torch.long),
                  "t_disrupt": torch.zeros(n_samples, device=DEVICE)}
    samples_clear = scheduler.sample(model, shape, cond_clear, DEVICE)

    # Disruption (class_id=1, t_disrupt=0.3 as example)
    cond_disrupt = {"class_id": torch.ones(n_samples, device=DEVICE, dtype=torch.long),
                    "t_disrupt": torch.full((n_samples,), 0.3, device=DEVICE)}
    samples_disrupt = scheduler.sample(model, shape, cond_disrupt, DEVICE)

samples_clear = samples_clear.cpu().numpy()
samples_disrupt = samples_disrupt.cpu().numpy()
print("Clear shape:", samples_clear.shape, "Disruption shape:", samples_disrupt.shape)

## Visualize: contrast Clear vs Disruption (2D image = channels × time)

In [ ]:
def plot_ecei_2d(ax, x, title, cmap="RdBu_r", vmin=None, vmax=None):
    # x: (1, 160, 7512) or (160, 7512)
    if x.ndim == 3:
        x = x.squeeze(0)
    if vmin is None:
        vmin = np.percentile(x, 2)
    if vmax is None:
        vmax = np.percentile(x, 98)
    im = ax.imshow(x, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax,
                   extent=[0, x.shape[1], x.shape[0], 0])
    ax.set_title(title)
    ax.set_xlabel("Time (decimated steps)")
    ax.set_ylabel("Channel")
    plt.colorbar(im, ax=ax)
    return im

fig, axes = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))
if n_samples == 1:
    axes = axes[:, None]
vmin = min(samples_clear.min(), samples_disrupt.min())
vmax = max(samples_clear.max(), samples_disrupt.max())

for i in range(n_samples):
    plot_ecei_2d(axes[0, i], samples_clear[i], f"Generated CLEAR (sample {i+1})", vmin=vmin, vmax=vmax)
for i in range(n_samples):
    plot_ecei_2d(axes[1, i], samples_disrupt[i], f"Generated DISRUPTION (sample {i+1})", vmin=vmin, vmax=vmax)

plt.suptitle("Diffusion: Clear vs Disruption (AdaLN conditioned)", fontsize=12)
plt.tight_layout()
plt.savefig("diffusion_clear_vs_disruption.png", dpi=150, bbox_inches="tight")
plt.show()

## Optional: compare with real data from dataset

In [ ]:
MMAP_DIR = "./subseqs_original_mmap"  # or path from config
DECIMATE = 10

ds = DecimatedEceiMmapDataset(MMAP_DIR, decimate_factor=DECIMATE, split="train")
real_clear, real_disrupt = [], []
for i in range(min(500, len(ds))):
    x, c, t = ds[i]
    x = x.numpy()
    x = (x - x.mean()) / (x.std() + 1e-5)
    if c == 0:
        real_clear.append(x)
    else:
        real_disrupt.append(x)
    if len(real_clear) >= n_samples and len(real_disrupt) >= n_samples:
        break

fig2, ax2 = plt.subplots(2, n_samples, figsize=(4 * n_samples, 8))
if n_samples == 1:
    ax2 = ax2[:, None]
for i in range(n_samples):
    if i < len(real_clear):
        plot_ecei_2d(ax2[0, i], real_clear[i], f"Real CLEAR {i+1}")
for i in range(n_samples):
    if i < len(real_disrupt):
        plot_ecei_2d(ax2[1, i], real_disrupt[i], f"Real DISRUPTION {i+1}")
plt.suptitle("Real data: Clear vs Disruption")
plt.tight_layout()
plt.show()

## Effect of t_disrupt (disruption only)

In [ ]:
t_values = [0.0, 0.25, 0.5, 0.75]
n_per = 2
fig3, ax3 = plt.subplots(len(t_values), n_per, figsize=(4 * n_per, 4 * len(t_values)))

with torch.no_grad():
    for row, t_d in enumerate(t_values):
        cond = {"class_id": torch.ones(n_per, device=DEVICE, dtype=torch.long),
                "t_disrupt": torch.full((n_per,), t_d, device=DEVICE)}
        out = scheduler.sample(model, (n_per, 1, 160, 7512), cond, DEVICE)
        out = out.cpu().numpy()
        for col in range(n_per):
            plot_ecei_2d(ax3[row, col], out[col], f"Disruption t_disrupt={t_d}")

plt.suptitle("Generated disruption at different t_disrupt (t_disruption - 300ms norm.)")
plt.tight_layout()
plt.show()